# Notebook 3: Data Sampling and Class Balancing

This notebook creates balanced subsets of data grouped by antibiotics to ensure representative sampling for model training.

---


In [3]:
import pandas as pd

df = pd.read_csv("../data/processed/refined_data.csv")

# Explicitly collect sampled dataframes and concatenate
sampled_groups = []
for _, group in df.groupby(["Antibiotic", "Resistant Phenotype"]):
    sampled_groups.append(group.sample(min(200, len(group)), random_state=42))

# Concatenate all sampled groups and reset the index
subset = pd.concat(sampled_groups).reset_index(drop=True)

print(subset.groupby(["Antibiotic", "Resistant Phenotype"]).size())
print(f"\nTotal genomes to download: {subset['Genome ID'].nunique()}")

subset["Genome ID"].drop_duplicates().to_csv("../data/interim/genome_ids.txt", index=False, header=False)

Antibiotic                     Resistant Phenotype
amikacin                       Intermediate            94
                               Resistant               41
                               Susceptible            200
amoxicillin                    Intermediate            77
                               Resistant              200
                                                     ... 
trimethoprim                   Susceptible            200
trimethoprim/sulfamethoxazole  Intermediate            41
                               Resistant              200
                               Susceptible            200
vancomycin                     Resistant                1
Length: 188, dtype: int64

Total genomes to download: 5459


In [4]:
df = pd.read_csv("../data/processed/refined_data.csv")

subset = df.groupby("Antibiotic").apply(
    lambda x: x.sample(min(200, len(x)), random_state=42)
).reset_index(level=0, drop=False)

print(subset["Antibiotic"].value_counts())
print(f"Total: {subset['Genome ID'].nunique()} genomes")

subset["Genome ID"].drop_duplicates() \
    .astype(str).to_csv("../data/interim/genome_ids_600.txt", index=False, header=False)

Antibiotic
amikacin                       200
amoxicillin                    200
amoxicillin/clavulanic acid    200
ampicillin                     200
ampicillin/sulbactam           200
                              ... 
linezolid                        1
sulbactam                        1
teicoplanin                      1
temocillin                       1
vancomycin                       1
Name: count, Length: 77, dtype: int64
Total: 4024 genomes


### Cell `4f5d2066` Output Explained:

This cell aims to create a balanced subset of data by sampling from each unique combination of 'Antibiotic' and 'Resistant Phenotype'.

**Code Functionality:**
1.  It reads the `refined_data.csv` into a DataFrame `df`.
2.  It then groups this DataFrame by both 'Antibiotic' and 'Resistant Phenotype'.
3.  For each of these groups, it samples a maximum of 200 rows (or fewer if the group has less than 200 rows) using `random_state=42` for reproducibility.
4.  The sampled data is stored in a new DataFrame called `subset`.

**Output Interpretation:**
*   **`print(subset.groupby(["Antibiotic", "Resistant Phenotype"]).size())`**: This output shows the number of samples obtained for each unique combination of 'Antibiotic' and 'Resistant Phenotype'. You can see that for most combinations, it tries to get 200 samples, but for some (e.g., 'amikacin' - 'Intermediate' has 94, 'vancomycin' - 'Resistant' has 1), the available data was less than 200, so it sampled all available rows.
*   **`print(f"\nTotal genomes to download: {subset['Genome ID'].nunique()}")`**: This indicates that after this specific sampling strategy, there are **5459 unique genome IDs** that have been selected. These IDs are then saved to `../data/interim/genome_ids.txt`.

### Cell `040c53a5` Output Explained:

This cell also creates a sampled subset, but its grouping strategy is different; it focuses on balancing the number of samples per antibiotic, without considering the resistance phenotype at this stage.

**Code Functionality:**
1.  Similar to the previous cell, it reads `refined_data.csv` into a DataFrame `df`.
2.  However, this time it groups the DataFrame only by 'Antibiotic'.
3.  For each 'Antibiotic' group, it samples a maximum of 200 rows (`min(200, len(x))`) with `random_state=42`.
4.  The resulting sampled data is stored in the `subset` DataFrame (overwriting the previous one).

**Output Interpretation:**
*   **`print(subset["Antibiotic"].value_counts())`**: This output shows the number of samples chosen for each individual 'Antibiotic'. You can see that many antibiotics have 200 samples, while others (like 'linezolid', 'temocillin', 'vancomycin') have fewer, indicating that there were fewer than 200 records available for those specific antibiotics.
*   **`print(f"Total: {subset['Genome ID'].nunique()} genomes")`**: This reports that this sampling method resulted in **4024 unique genome IDs**. These IDs are then saved to `../data/interim/genome_ids_600.txt`.